In [ ]:
# =============================================================================
# Project 2 (RAG), step 1: long-document corpus + chunking sweep with hybrid retrieval
# Run in Google Colab (T4). Paste each "# %% CELL" block into its own cell, in order.
# This notebook is self-contained: it does not need the Project 1 variables.
# =============================================================================

# %% CELL P2-1 -- install + imports
!pip install -q datasets sentence-transformers rank_bm25 pandas
import numpy as np

In [ ]:
# %% CELL P2-2 -- pure helpers (no model, no dataset): offsets, chunking, gold labels, eval
import re, time, random

def assemble_doc(paras):
    """Join paragraphs into one long document and return the char offset where each starts."""
    offsets, pos = [], 0
    for p in paras:
        offsets.append(pos)
        pos += len(p) + 2            # +2 for the "\n\n" separator
    return "\n\n".join(paras), offsets

WORD_RE = re.compile(r"\S+")

def chunk_words_with_spans(text, chunk_size=100, overlap=20):
    """Sliding window over words. Returns [(start_char, end_char), ...] per chunk.
    Stops as soon as a window reaches the last word, so there is no redundant tail chunk."""
    assert 0 <= overlap < chunk_size
    spans = [m.span() for m in WORD_RE.finditer(text)]
    step = chunk_size - overlap
    chunks, i = [], 0
    while i < len(spans):
        j = min(i + chunk_size, len(spans))
        chunks.append((spans[i][0], spans[j - 1][1]))
        if j == len(spans):
            break
        i += step
    return chunks

def build_chunks(docs, chunk_size, overlap):
    chunks = []
    for doc_id, text in docs.items():
        for k, (s, e) in enumerate(chunk_words_with_spans(text, chunk_size, overlap)):
            chunks.append({"id": f"{doc_id}#{k}", "doc_id": doc_id,
                           "start": s, "end": e, "text": text[s:e]})
    return chunks

def attach_gold(questions, chunks):
    """Gold = every chunk that FULLY contains the answer span. If none does (the answer was
    split across a boundary), gold is empty and the question counts as a miss."""
    by_doc = {}
    for c in chunks:
        by_doc.setdefault(c["doc_id"], []).append(c)
    out = []
    for q in questions:
        gold = {c["id"] for c in by_doc[q["doc_id"]]
                if c["start"] <= q["start"] and q["end"] <= c["end"]}
        out.append({**q, "gold": gold})
    return out

def evaluate(retrieve_fn, questions, k_values=(1, 5, 10)):
    hits = {k: [] for k in k_values}
    rr, lat = [], []
    for q in questions:
        t0 = time.perf_counter()
        ranked = retrieve_fn(q["question"], top_k=max(k_values))
        lat.append(time.perf_counter() - t0)
        gold = q["gold"]
        for k in k_values:
            hits[k].append(1 if any(c in gold for c in ranked[:k]) else 0)
        rank = next((i + 1 for i, c in enumerate(ranked) if c in gold), None)
        rr.append(1.0 / rank if rank else 0.0)
    lat.sort()
    out = {f"recall@{k}": sum(v) / len(v) for k, v in hits.items()}
    out["mrr"] = sum(rr) / len(rr)
    out["p50_ms"] = lat[len(lat) // 2] * 1000
    out["p95_ms"] = lat[int(len(lat) * 0.95)] * 1000
    return out

# --- retrieval helpers carried over from Project 1 ---
STOPWORDS = {"the","a","an","of","in","on","at","to","is","was","were","are","and","or",
             "for","by","with","as","what","which","who","when","where","how","did","does",
             "do","from","that","this","it","its","be","been","has","have","had"}

def better_tokenize(text):
    return [t for t in re.findall(r"\w+", text.lower()) if t not in STOPWORDS]

def rrf_fuse(ranked_lists, weights, k_constant=60, top_k=10):
    scores = {}
    for ranked, w in zip(ranked_lists, weights):
        for rank, pid in enumerate(ranked):
            scores[pid] = scores.get(pid, 0.0) + w / (k_constant + rank + 1)
    return sorted(scores, key=lambda pid: -scores[pid])[:top_k]

In [ ]:
# %% CELL P2-3 -- load SQuAD v2 articles as LONG documents, with answer offsets
from datasets import load_dataset

def build_corpus(n_articles=150, n_questions=500, seed=42):
    ds = load_dataset("rajpurkar/squad_v2", split="train")
    by_title = {}
    for row in ds:
        by_title.setdefault(row["title"], []).append(row)

    rng = random.Random(seed)
    titles = rng.sample(sorted(by_title), n_articles)

    docs, questions, skipped = {}, [], 0
    for t_i, title in enumerate(titles):
        rows = by_title[title]
        paras, pidx = [], {}
        for r in rows:
            if r["context"] not in pidx:
                pidx[r["context"]] = len(paras)
                paras.append(r["context"])
        text, offsets = assemble_doc(paras)
        doc_id = f"d{t_i}"
        docs[doc_id] = text

        for r in rows:
            if not r["answers"]["text"]:          # unanswerable: kept for the abstain-gate step later
                continue
            a_text = r["answers"]["text"][0]
            start = offsets[pidx[r["context"]]] + r["answers"]["answer_start"][0]
            end = start + len(a_text)
            if text[start:end] != a_text:         # offset sanity check; skip anything inconsistent
                skipped += 1
                continue
            questions.append({"question": r["question"], "doc_id": doc_id,
                              "start": start, "end": end,
                              "answers": list(r["answers"]["text"])})

    questions = rng.sample(questions, min(n_questions, len(questions)))
    words = sum(len(t.split()) for t in docs.values())
    print(f"{len(docs)} documents, avg {words // len(docs)} words each; "
          f"{len(questions)} questions (skipped {skipped} with bad offsets)")
    return docs, questions

docs, questions = build_corpus()

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

150 documents, avg 5126 words each; 500 questions (skipped 0 with bad offsets)


In [ ]:
# %% CELL P2-4 -- chunking sweep: chunk size x overlap, dense vs hybrid
import pandas as pd
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

model = SentenceTransformer("BAAI/bge-small-en-v1.5")   # max 512 tokens: keep chunks <= ~300 words

def build_retrievers(chunks):
    ids = [c["id"] for c in chunks]
    texts = [c["text"] for c in chunks]
    emb = model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    bm25 = BM25Okapi([better_tokenize(t) for t in texts])

    def dense(q, top_k=10):
        qe = model.encode([q], normalize_embeddings=True)[0]
        return [ids[i] for i in np.argsort(-(emb @ qe))[:top_k]]

    def sparse(q, top_k=10):
        s = bm25.get_scores(better_tokenize(q))
        return [ids[i] for i in np.argsort(-s)[:top_k]]

    def hybrid(q, top_k=10):
        return rrf_fuse([dense(q, 50), sparse(q, 50)], [1.0, 1.0], top_k=top_k)

    return dense, hybrid

rows = []
for size in (50, 100, 200, 300):
    for frac in (0.0, 0.2):
        overlap = int(size * frac)
        chunks = build_chunks(docs, size, overlap)
        qs = attach_gold(questions, chunks)
        split_pct = 100 * sum(1 for q in qs if not q["gold"]) / len(qs)
        dense, hybrid = build_retrievers(chunks)
        for name, fn in (("dense", dense), ("hybrid", hybrid)):
            res = evaluate(fn, qs)
            rows.append({"chunk_words": size, "overlap": overlap, "n_chunks": len(chunks),
                         "answer_split_%": round(split_pct, 1), "retriever": name, **res})
            print(f"size={size:3d} overlap={overlap:3d} {name:6s} "
                  f"r@5={res['recall@5']:.3f} split={split_pct:.1f}%")

df = pd.DataFrame(rows).round(3)
print(df.to_string(index=False))
df.to_csv("project2_chunking_sweep.csv", index=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# %% CELL P2-5 -- install (active line, run it) + config + check step-1 state
!pip install -q sentence-transformers rank_bm25 pandas

CHUNK_SIZE, OVERLAP = 100, 20     # my pick: the step-1 sweep never finished. Change after you see its table.
POOLS = (10, 20, 50)              # how many hybrid candidates the re-ranker sees
RERANKERS = {                     # name -> model id
    "minilm-l6":         "cross-encoder/ms-marco-MiniLM-L-6-v2",   # small, fast
    "bge-reranker-base": "BAAI/bge-reranker-base",                 # larger, usually stronger
}

needed = ["docs", "questions", "build_chunks", "attach_gold", "evaluate", "rrf_fuse", "better_tokenize"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(f"Missing {missing}. Run step-1 cells P2-1, P2-2, P2-3 first (same session).")
print("step-1 state found:", len(docs), "documents,", len(questions), "questions")


step-1 state found: 150 documents, 500 questions


In [ ]:
# %% CELL P2-6 -- re-rank logic (pure functions: no model, no dataset)
def rerank_ids(question, candidate_ids, id2text, score_fn, top_k=10):
    """Score every (question, chunk) pair with score_fn, return candidate ids best-first."""
    if not candidate_ids:
        return []
    pairs = [(question, id2text[c]) for c in candidate_ids]
    scores = score_fn(pairs)
    order = sorted(range(len(candidate_ids)), key=lambda i: -scores[i])
    return [candidate_ids[i] for i in order[:top_k]]

def make_reranked(candidate_fn, pool, id2text, score_fn):
    """retrieve_fn(question, top_k) -> ids. Takes `pool` candidates, re-ranks, returns top_k."""
    def retrieve(question, top_k=10):
        candidates = candidate_fn(question, top_k=pool)
        return rerank_ids(question, candidates, id2text, score_fn, top_k=top_k)
    return retrieve

In [ ]:
# %% CELL P2-7 -- build the hybrid retriever, then run baseline vs re-rankers
import time
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

chunks = build_chunks(docs, CHUNK_SIZE, OVERLAP)
qs = attach_gold(questions, chunks)
ids = [c["id"] for c in chunks]
texts = [c["text"] for c in chunks]
id2text = dict(zip(ids, texts))
miss = 100 * sum(1 for q in qs if not q["gold"]) / len(qs)
print(f"{len(chunks)} chunks (size={CHUNK_SIZE}, overlap={OVERLAP}); {miss:.1f}% of questions have no single gold chunk")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
bm25 = BM25Okapi([better_tokenize(t) for t in texts])

def dense(q, top_k=10):
    qe = embedder.encode([q], normalize_embeddings=True)[0]
    return [ids[i] for i in np.argsort(-(emb @ qe))[:top_k]]

def sparse(q, top_k=10):
    return [ids[i] for i in np.argsort(-bm25.get_scores(better_tokenize(q)))[:top_k]]

def hybrid(q, top_k=10):
    return rrf_fuse([dense(q, 50), sparse(q, 50)], [1.0, 1.0], top_k=top_k)

KS = (1, 3, 5, 10)
rows = []

base = evaluate(hybrid, qs, k_values=KS)
rows.append({"config": "hybrid (no re-rank)", "pool": "-", **base})

# Ceiling: a re-ranker can only reorder what the pool contains, so recall@pool is the best it can do.
print("\nCeiling (hybrid recall@pool):")
for pool in POOLS:
    ceil = evaluate(hybrid, qs, k_values=(pool,))[f"recall@{pool}"]
    print(f"  recall@{pool} = {ceil:.3f}")

for name, model_id in RERANKERS.items():
    ce = CrossEncoder(model_id, max_length=512)
    score_fn = lambda pairs, ce=ce: ce.predict(pairs, batch_size=32, show_progress_bar=False)
    for q in qs[:3]:                                   # warm-up so GPU start-up is not in the timings
        score_fn([(q["question"], texts[0])] * 8)
    for pool in POOLS:
        fn = make_reranked(hybrid, pool, id2text, score_fn)
        res = evaluate(fn, qs, k_values=KS)
        rows.append({"config": f"hybrid + {name}", "pool": pool, **res})
        print(f"{name:18s} pool={pool:2d}  r@1={res['recall@1']:.3f}  r@5={res['recall@5']:.3f}  "
              f"mrr={res['mrr']:.3f}  p50={res['p50_ms']:.0f}ms")

df = pd.DataFrame(rows).round(3)
print("\n", df.to_string(index=False))
df.to_csv("project2_reranker.csv", index=False)

9652 chunks (size=100, overlap=20); 0.0% of questions have no single gold chunk


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Ceiling (hybrid recall@pool):
  recall@10 = 0.916
  recall@20 = 0.956
  recall@50 = 0.982


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

... (13 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

bge-reranker-base  pool=10  r@1=0.488  r@5=0.820  mrr=0.629  p50=149ms
bge-reranker-base  pool=20  r@1=0.416  r@5=0.750  mrr=0.550  p50=263ms
bge-reranker-base  pool=50  r@1=0.340  r@5=0.600  mrr=0.455  p50=613ms

                     config pool  recall@1  recall@3  recall@5  recall@10   mrr  p50_ms  p95_ms
       hybrid (no re-rank)    -     0.616     0.788     0.844      0.916 0.717  39.539  58.671
        hybrid + minilm-l6   10     0.770     0.880     0.902      0.916 0.828  77.898 100.514
        hybrid + minilm-l6   20     0.782     0.894     0.924      0.948 0.844  98.985 126.388
        hybrid + minilm-l6   50     0.790     0.902     0.932      0.958 0.853 140.772 174.126
hybrid + bge-reranker-base   10     0.488     0.728     0.820      0.916 0.629 149.176 178.527
hybrid + bge-reranker-base   20     0.416     0.614     0.750      0.884 0.550 263.086 315.239
hybrid + bge-reranker-base   50     0.340     0.504     0.600      0.748 0.455 613.186 691.500


In [ ]:
from google.colab import files
files.download("project2_reranker.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =============================================================================
# Project 2 (RAG), step 3: grounded generation with citations, using an open model
# Run in Colab on a T4 GPU runtime. Needs step-1 cells P2-1, P2-2, P2-3 already run in
# this session (docs, questions, build_chunks, attach_gold, rrf_fuse, better_tokenize).
# It rebuilds its own retrieval and loads MiniLM itself, so it does not depend on step 2's
# variables (step 2 left the weaker bge-reranker in `ce` / `score_fn`).
# Cells: P2-8 setup | P2-9 helpers (no GPU) | P2-10 load models + smoke test | P2-11 full run
# =============================================================================

# %% CELL P2-8 -- install (active line, run it) + config + check step-1 state
!pip install -q accelerate transformers sentence-transformers rank_bm25 pandas

MODEL_ID = "microsoft/Phi-3.5-mini-instruct"   # MIT license, ~7.6 GB in fp16, fits a T4.
# If it fails to load or generates garbage in fp16, try "Qwen/Qwen2.5-3B-Instruct" (check its license first).
RERANKER_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHUNK_SIZE, OVERLAP = 100, 20   # same setup as step 2
POOL, TOP_K = 20, 5             # re-rank 20 candidates, give the LLM the best 5
N_EVAL = 100                    # questions per mode; raise later if you want tighter numbers
MAX_NEW = 48                    # answers are a few words plus citations

needed = ["docs", "questions", "build_chunks", "attach_gold", "rrf_fuse", "better_tokenize"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(f"Missing {missing}. Run step-1 cells P2-1, P2-2, P2-3 first (same session).")
print("step-1 state found:", len(docs), "documents,", len(questions), "questions")

step-1 state found: 150 documents, 500 questions


In [ ]:
# %% CELL P2-9 -- helpers: prompts, answer/citation parsing, metrics (pure Python, no GPU)
import re, string, collections, random, time, json

PUNCT = set(string.punctuation)

def normalize_answer(s):
    """SQuAD-style normalization: lowercase, drop punctuation and articles, collapse spaces."""
    s = "".join(ch for ch in s.lower() if ch not in PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    n = sum((collections.Counter(p) & collections.Counter(g)).values())
    if n == 0:
        return 0.0
    prec, rec = n / len(p), n / len(g)
    return 2 * prec * rec / (prec + rec)

def best_over_golds(metric, pred, golds):
    return max(metric(pred, g) for g in golds) if golds else 0.0

CITE_RE = re.compile(r"\[([^\[\]]+)\]")
CITE_TOKEN_RE = re.compile(r"^d\d+#\S*$")      # anything that tries to be a chunk id; real ones look like d12#3
IDK_RE = re.compile(r"^\W*i\s*(?:do\s*not|don'?t|dont)\s*know", re.I)

def parse_answer(text):
    """Split model output into the answer (first non-empty line, citations removed),
    the chunk ids it cited, and whether it abstained."""
    text = text.replace("’", "'")
    cited = []
    for m in CITE_RE.finditer(text):
        cited += [t for t in re.split(r"[,;\s]+", m.group(1)) if CITE_TOKEN_RE.match(t)]
    cited = list(dict.fromkeys(cited))                  # drop repeats, keep order
    cleaned = CITE_RE.sub("", text)
    lines = [l.strip() for l in cleaned.splitlines() if l.strip()]
    answer = lines[0] if lines else ""
    idk = bool(IDK_RE.match(answer))
    return {"answer": "" if idk else answer.strip(" .,;\t"), "cited": cited, "idk": idk}

def build_prompt(question, chunk_ids, id2text, closed_book=False):
    if closed_book:
        return ("Answer the question with the shortest answer that fully answers it: "
                "a few words, not a sentence.\n\n"
                f"Question: {question}\nAnswer:")
    ctx = "\n\n".join(f"[{cid}] {id2text[cid]}" for cid in chunk_ids)
    return ("Answer the question using ONLY the context chunks below.\n"
            "Rules:\n"
            "- Reply with the shortest answer that fully answers the question: a few words, not a sentence.\n"
            "- After the answer, cite the id of each chunk you used in square brackets, like [d12#3].\n"
            "- Output only the answer and the citations, with no explanation or notes.\n"
            "- If the chunks do not contain the answer, reply exactly: I don't know.\n\n"
            f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:")

def score_one(raw_text, golds, provided_ids, gold_ids):
    parsed = parse_answer(raw_text)
    ans, cited = parsed["answer"], parsed["cited"]
    provided, gold_ids = set(provided_ids), set(gold_ids)
    return {
        "answer": ans, "idk": parsed["idk"], "cited": cited,
        "em": best_over_golds(exact_match, ans, golds),
        "f1": best_over_golds(f1_score, ans, golds),
        "hit": bool(provided & gold_ids),                      # was a correct chunk given to the LLM?
        "has_cite": bool(cited),
        "cite_valid": bool(cited) and all(c in provided for c in cited),   # no invented chunk ids
        "cite_correct": any(c in gold_ids for c in cited if c in provided), # cited a chunk with the answer
    }

def _mean(xs):
    xs = [float(x) for x in xs]
    return sum(xs) / len(xs) if xs else None

def summarize(recs, closed_book=False):
    n = len(recs)
    out = {"n": n, "EM": _mean(r["em"] for r in recs), "F1": _mean(r["f1"] for r in recs),
           "idk_rate": _mean(r["idk"] for r in recs)}
    if not closed_book:
        hit = [r for r in recs if r["hit"]]
        miss = [r for r in recs if not r["hit"]]
        out["retrieval_hit"] = len(hit) / n
        out["F1_if_hit"] = _mean(r["f1"] for r in hit)
        out["F1_if_miss"] = _mean(r["f1"] for r in miss)
        out["idk_if_miss"] = _mean(r["idk"] for r in miss)
        answered = [r for r in recs if not r["idk"]]
        out["cite_present"] = _mean(r["has_cite"] for r in answered)
        with_cite = [r for r in answered if r["has_cite"]]
        out["cite_valid"] = _mean(r["cite_valid"] for r in with_cite)
        out["cite_correct"] = _mean(r["cite_correct"] for r in with_cite)
    out["prompt_tokens"] = _mean(r["prompt_tokens"] for r in recs)
    gs = sorted(r["gen_s"] for r in recs)
    out["p50_gen_s"] = gs[len(gs) // 2]
    return out

def rerank_ids(question, candidate_ids, id2text, score_fn, top_k=10):
    """Same function as step 2: score every (question, chunk) pair, return ids best-first."""
    if not candidate_ids:
        return []
    scores = score_fn([(question, id2text[c]) for c in candidate_ids])
    order = sorted(range(len(candidate_ids)), key=lambda i: -scores[i])
    return [candidate_ids[i] for i in order[:top_k]]

In [ ]:
# %% CELL P2-10 -- build retrieval, load models, run a 3-question smoke test
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- retrieval (same recipe as step 2) ---
chunks = build_chunks(docs, CHUNK_SIZE, OVERLAP)
qs_all = attach_gold(questions, chunks)
ids = [c["id"] for c in chunks]
texts = [c["text"] for c in chunks]
id2text = dict(zip(ids, texts))
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
bm25 = BM25Okapi([better_tokenize(t) for t in texts])

def hybrid(q, top_k=10):
    qe = embedder.encode([q], normalize_embeddings=True)[0]
    dense_ids = [ids[i] for i in np.argsort(-(emb @ qe))[:50]]
    sparse_ids = [ids[i] for i in np.argsort(-bm25.get_scores(better_tokenize(q)))[:50]]
    return rrf_fuse([dense_ids, sparse_ids], [1.0, 1.0], top_k=top_k)

ce = CrossEncoder(RERANKER_ID, max_length=512)      # MiniLM, explicitly (not step 2's leftover bge)
score_fn = lambda pairs: ce.predict(pairs, batch_size=32, show_progress_bar=False)

# --- the LLM ---
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()

def generate(prompt):
    msgs = [{"role": "user", "content": prompt}]       # instructions live in the user turn (works for any chat template)
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    n_in = inputs["input_ids"].shape[1]
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
    dt = time.perf_counter() - t0
    return tok.decode(out[0, n_in:], skip_special_tokens=True), n_in, dt

def provide_chunks(mode, question):
    if mode == "closed-book":
        return []
    if mode == "hybrid top-5":
        return hybrid(question, top_k=TOP_K)
    cands = hybrid(question, top_k=POOL)               # "hybrid + MiniLM top-5"
    return rerank_ids(question, cands, id2text, score_fn, top_k=TOP_K)

MODES = ["closed-book", "hybrid top-5", "hybrid + MiniLM top-5"]
sample = random.Random(0).sample(qs_all, N_EVAL)

generate("Say hi.")                                    # warm-up so GPU start-up isn't in the timings
print("\n--- smoke test: 3 questions, re-ranked mode. Read these before running P2-11. ---")
for item in sample[:3]:
    chosen = provide_chunks("hybrid + MiniLM top-5", item["question"])
    prompt = build_prompt(item["question"], chosen, id2text)
    raw, n_in, dt = generate(prompt)
    print(f"\nQ: {item['question']}\ngold: {item['answers'][:2]}\nprompt tokens: {n_in}  time: {dt:.1f}s\nraw output: {raw!r}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (26 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]


--- smoke test: 3 questions, re-ranked mode. Read these before running P2-11. ---

Q: Who became president of Russia after Gorbachev left office?
gold: ['Boris Yeltsin']
prompt tokens: 942  time: 1.3s
raw output: 'Boris Yeltsin [d31#98][d31#97]'

Q: Palermo is the home of what mafia?
gold: ['the Sicilian Mafia']
prompt tokens: 984  time: 1.5s
raw output: 'Sicilian Mafia [d125#36#3] [d125#3#3]'

Q: In what year did Schwarzenegger and Maria Shriver separate?
gold: ['2011']
prompt tokens: 1052  time: 2.4s
raw output: '2011 [d140#92], [d140#91]\n\nNote: The answer is derived from the context chunks [d140#92] and [d'


In [ ]:
for item in sample[:3]:
    chosen = provide_chunks("hybrid + MiniLM top-5", item["question"])
    raw, n_in, dt = generate(build_prompt(item["question"], chosen, id2text))
    print(repr(raw), f"({dt:.1f}s)")

'Boris Yeltsin [d31#98][d31#97]' (1.7s)
'Sicilian Mafia [d125#36#3] [d125#3#3]' (1.8s)
'2011 [d140#92], [d140#91]\n\nNote: The answer is derived from the context chunks [d140#92] and [d' (2.4s)


In [ ]:
# %% CELL P2-11 -- full run: 3 modes x N_EVAL questions
all_recs, summaries, t_start = {}, {}, time.time()
with open("project2_generations.jsonl", "w") as f:
    for mode in MODES:
        recs = []
        for k, item in enumerate(sample):
            chosen = provide_chunks(mode, item["question"])
            prompt = build_prompt(item["question"], chosen, id2text, closed_book=(mode == "closed-book"))
            raw, n_in, dt = generate(prompt)
            rec = score_one(raw, item["answers"], chosen, item["gold"])
            rec.update({"prompt_tokens": n_in, "gen_s": dt})
            recs.append(rec)
            f.write(json.dumps({"mode": mode, "question": item["question"], "golds": item["answers"],
                                "provided": chosen, "raw": raw, **rec}) + "\n")
            if (k + 1) % 25 == 0:
                print(f"{mode}: {k + 1}/{N_EVAL}  ({(time.time() - t_start) / 60:.1f} min elapsed)")
        all_recs[mode] = recs
        summaries[mode] = summarize(recs, closed_book=(mode == "closed-book"))

df = pd.DataFrame(summaries).T.round(3)
print("\n", df.to_string())
df.to_csv("project2_generation.csv")
print("\nSaved project2_generation.csv and project2_generations.jsonl (per-question outputs, reused in the Evals project).")

closed-book: 25/100  (0.1 min elapsed)
closed-book: 50/100  (0.3 min elapsed)
closed-book: 75/100  (0.5 min elapsed)
closed-book: 100/100  (0.6 min elapsed)
hybrid top-5: 25/100  (1.3 min elapsed)
hybrid top-5: 50/100  (2.2 min elapsed)
hybrid top-5: 75/100  (3.0 min elapsed)
hybrid top-5: 100/100  (3.9 min elapsed)
hybrid + MiniLM top-5: 25/100  (4.7 min elapsed)
hybrid + MiniLM top-5: 50/100  (5.6 min elapsed)
hybrid + MiniLM top-5: 75/100  (6.4 min elapsed)
hybrid + MiniLM top-5: 100/100  (7.3 min elapsed)

                            n    EM     F1  idk_rate  prompt_tokens  p50_gen_s  retrieval_hit  F1_if_hit  F1_if_miss  idk_if_miss  cite_present  cite_valid  cite_correct
closed-book            100.0  0.10  0.242       0.0          45.87      0.309            NaN        NaN         NaN          NaN           NaN         NaN           NaN
hybrid top-5           100.0  0.58  0.766       0.0         960.76      2.310           0.91      0.821       0.201          0.0           1.0   

# New Section